In [ ]:
import joblib
import pandas as pd
import numpy as np

# --- 1. SETUP: SIC MAPPING (The "Mini-NLP" Layer) ---
# In a real app, this would be a full semantic search or database.
# For this prototype, we map common keywords to the correct UK SIC Codes.
SIC_MAPPING = {
    "developer": "62020",    # Computer consultancy
    "engineer": "62020",     # Computer consultancy
    "data": "62020",         # Computer consultancy
    "nurse": "86101",        # Hospital activities
    "doctor": "86101",       # Hospital activities
    "surgeon": "86101",      # Hospital activities
    "teacher": "85310",      # General secondary education
    "educator": "85310",     # General secondary education
    "banker": "64191",       # Banks
    "finance": "64999",      # Financial services
    "accountant": "69201",   # Accounting/Auditing
    "sales": "47190",        # Retail
    "cleaner": "81210",      # General cleaning
    "ceo": "70100",          # Head office activities
    "manager": "70100"       # Head office activities
}

def get_sic_from_title(title):
    """
    Simple keyword matching to find SIC code.
    Defaults to '70100' (Head Office) if no match found.
    """
    title = title.lower()
    for keyword, code in SIC_MAPPING.items():
        if keyword in title:
            return code
    return "70100" # Default fallback

# --- 2. INFERENCE FUNCTION ---
def predict_bias(job_title, employer_size="250 to 499"):
    # A. Load the Model
    try:
        model = joblib.load('bias_predictor_uk.pkl')
    except FileNotFoundError:
        return "Error: Model file 'bias_predictor_uk.pkl' not found. Run training first."

    # B. Prepare the Input Data
    # We must match the EXACT columns the model was trained on.

    # 1. Map Title -> SIC
    sic_code = get_sic_from_title(job_title)

    # 2. Construct DataFrame
    input_data = {
        "EmployerSize": [employer_size],
        "SicCodes": [sic_code],
        # The model expects quartile data. Since we don't have it for a resume,
        # we pass 0.0 (neutral) or the industry average if we had it.
        "MaleTopQuartile": [0.0],
        "FemaleTopQuartile": [0.0],
        "MaleUpperMiddleQuartile": [0.0],
        "FemaleUpperMiddleQuartile": [0.0],
        "MaleLowerMiddleQuartile": [0.0],
        "FemaleLowerMiddleQuartile": [0.0],
        "MaleLowerQuartile": [0.0],
        "FemaleLowerQuartile": [0.0]
    }

    df = pd.DataFrame(input_data)

    # C. Run Prediction
    prediction = model.predict(df)[0]

    return float(prediction), sic_code

# --- 3. INTERACTIVE TEST ---
if __name__ == "__main__":
    print("=== Bias Prediction Engine (Prototype) ===")
    print("Type a job title to see the predicted UK Gender Pay Gap.")
    print("Type 'exit' to quit.\n")

    while True:
        user_input = input("Enter Job Title: ")
        if user_input.lower() == 'exit':
            break

        score, sic = predict_bias(user_input)

        print(f"--> Mapped to SIC Code: {sic}")
        print(f"--> Predicted Gender Pay Gap: {score:.2f}%")

        if score > 15.0:
            print("⚠️  Result: HIGH RISK of Disparity")
        elif score > 5.0:
            print("⚠️  Result: MODERATE RISK")
        else:
            print("✅  Result: LOW RISK")
        print("-" * 30)

=== Bias Prediction Engine (Prototype) ===
Type a job title to see the predicted UK Gender Pay Gap.
Type 'exit' to quit.

Enter Job Title: developer
--> Mapped to SIC Code: 62020
--> Predicted Gender Pay Gap: 21.10%
⚠️  Result: HIGH RISK of Disparity
------------------------------
Enter Job Title: doctor
--> Mapped to SIC Code: 86101
--> Predicted Gender Pay Gap: 21.05%
⚠️  Result: HIGH RISK of Disparity
------------------------------
Enter Job Title: nurse
--> Mapped to SIC Code: 86101
--> Predicted Gender Pay Gap: 21.05%
⚠️  Result: HIGH RISK of Disparity
------------------------------


# New section